In [1]:
import numpy as np

from coppertop.pipe import *
from coppertop.dm.core.types import offset
from coppertop.dm.core import kvs, first, values, drop, at, keys, shape, count, collect, pad, take
from coppertop.dm.numpy import shape
from coppertop.dm.core.types import pylist

from coppertop.dm.examples.cluedo.core import *
from coppertop.dm.examples.cluedo import simple, bayes_pad, reports
from coppertop.dm.examples.cluedo.bayes_pad import PP

from coppertop.dm.utils.module import unload
'coppertop.dm.examples.cluedo.games' >> unload

from coppertop.dm.examples.cluedo.games import games

In [2]:
'coppertop.dm.examples.cluedo.games' >> unload
from coppertop.dm.examples.cluedo.games import games
deal, preevents, events = games[9]

In [3]:
# deal = {Pl: [St, Ki, Ha, Ca, Or], Gr: 5, Or: 4, Pe: 4}

In [5]:
Me, hand = deal >> kvs >> first
handSizes = [len(hand)] + list(deal >> values >> drop >> 1)
knowns = [hand] + [[]] * (len(deal) - 1)
f'{deal >> keys | pylist}' >> PP;

[Pl, Gr, Or, Pe]


In [6]:
with context(PPEachPossibility=False):
    _.possMaster, slices = bayes_pad.createPossibilities(people, weapons, rooms, handSizes, knowns)

nTBI:  140  nEachSet:  90090   nCols:  16  mem:  201.8016MB
zeros: 0.00047854200000000004  =0:  0.027382625


In [14]:
_.pads = {}
_.events = []
_.suggestId = 0
_.otherPlayers = deal >> keys >> drop >> 1
_.rowTitles = ([TBI] + _.otherPlayers) >> collect >> (lambda x: repr(x) >> pad(_,{'left':5}))
_.knownCards = dict(zip(deal >> keys, knowns))
_.ss = dict(zip([TBI, Me] + _.otherPlayers, slices))
_.DEBUG = True

s0, s1, s2, s3 = 1.05, 0.5, 0.25, 0.0

_.poss = np.array(_.possMaster)
_.l = np.ones(_.poss >> shape >> at >> (0 | offset))
f'{(_.l >> count) * 8 / 1_000_000:,.1f} MB' >> PP;

100.9 MB


In [10]:
preevents = [[Pe, Le], [Or, Ba]]
preevents = []

In [12]:
Me, hand = deal >> kvs >> first
otherHandSizesById = deal >> drop >> Me

like = {0: 100, 1: 10, 2: 5, 3: 0}
event2 = events
helper = simple.createHelper(Me, hand, deal >> drop >> Me)
helper = helper >> simple.figureKnown >> preevents + event2
helper = helper >> simple.processResponses >> event2
# helper = helper >> processSuggestions1(_, _, like) >> event2
helper = helper >> simple.processSuggestions2(_, _, like) >> event2
helper >> reports.rep2 >> reports.PP;

840000
876000
110000
                                       Me 5        Green 5       Orchid 4     Peacock 4   
Green            X                        - b         a -          bf -           c -     
Mustard          -         75%            -             ?             ?             ?     
Orchid           -                        -             -             -           g X     
Peacock          -                        -           e X             -             -     
Plum             -         75%            -             ?             ?             ?     
Scarlet          -         9%          dh X             -             - d           ? h   
----                                               
Candlestick      -         75%            -             ?             ?             ?     
Dagger           -         41%          d X             -             ? d           ?     
Lead Pipe        -         75%            -             ?             ?             ?     
Revolver         

In [15]:
events >> bayes_pad.processEvents(_, s0, s1, s2, s3)

calcPad: #0, 6,526.9ms
0: 3827250,  1: 8558550,  2: 2438100,  3: 226800
old: 12612600, 201.8 MB
new: 12385800, 198.2 MB
7561260
old: 12385800, 198.2 MB
new: 4824540, 77.2 MB
3660300
old: 4824540, 77.2 MB
new: 1164240, 18.6 MB
0
unchanged: 1164240, 18.6 MB
calcPad: #1, 522.3ms
0: 1164240,  1: 0,  2: 0,  3: 0
unchanged: 1164240, 18.6 MB
0
unchanged: 1164240, 18.6 MB
0
unchanged: 1164240, 18.6 MB
calcPad: #2, 495.3ms
0: 800520,  1: 363720,  2: 0,  3: 0
unchanged: 1164240, 18.6 MB
0
unchanged: 1164240, 18.6 MB
958650
old: 1164240, 18.6 MB
new: 205590, 3.3 MB
85470
old: 205590, 3.3 MB
new: 120120, 1.9 MB
calcPad: #3, 51.9ms
0: 120120,  1: 0,  2: 0,  3: 0
unchanged: 120120, 1.9 MB
39900
old: 120120, 1.9 MB
new: 80220, 1.3 MB
42630
old: 80220, 1.3 MB
new: 37590, 0.6 MB
calcPad: #4, 15.4ms
0: 13650,  1: 23940,  2: 4620,  3: 0
unchanged: 37590, 0.6 MB
18186
old: 37590, 0.6 MB
new: 19404, 0.3 MB
13104
old: 19404, 0.3 MB
new: 6300, 0.1 MB
0
unchanged: 6300, 0.1 MB
calcPad: #5, 3.5ms
0: 0,  1: 630

In [19]:
n = 5
n >> bayes_pad.printBayesPad
'' >> PP
n+1 >> bayes_pad.printBayesPad
'' >> PP
n+2 >> bayes_pad.printBayesPad

        Gr     Mu     Or     Pe     Pl     Sc     Ca     Da     Le     Re     Ro     Wr     Ba     Bi     Co     Di     Ha     Ki     Li     Lo     St     
TBI   100.0%    -      -      -      -      -      -      -      -   100.0%    -      -      -     3.8%   3.8%  18.2%   3.8%  24.2%    -    46.1%    -  
Gr       -    30.4%  30.4% 100.0%  30.4%    -    30.4%    -    30.4%    -    30.4%    -      -    29.4%  29.4%    -    29.4%  75.8%    -    53.9%    -  
Or       -    33.8%  33.8%    -    33.8%    -    33.8%    -    33.8%    -    33.8%    -      -    32.4%  32.4%    -    32.4%    -   100.0%    -      -  
Pe       -    35.8%  35.8%    -    35.8%    -    35.8%    -    35.8%    -    35.8%    -      -    34.4%  34.4%  81.8%  34.4%    -      -      -      -  

        Gr     Mu     Or     Pe     Pl     Sc     Ca     Da     Le     Re     Ro     Wr     Ba     Bi     Co     Di     Ha     Ki     Li     Lo     St     
TBI   100.0%    -      -      -      -      -      -      -      -   100.0%